# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [7]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


I chose a line chart because it clearly shows how population growth changed across continents over time. I used percentage growth to make the comparison fair, highlighted Africa in blue, and added direct labels to make the chart easier to read.

In [9]:
continent_pop = df_long.groupby(
    ['Continent', 'Year'], as_index=False
)['Population'].sum()

base_pop = continent_pop[
    continent_pop['Year'] == 1970
][['Continent', 'Population']].rename(
    columns={'Population': 'Population_1970'}
)

continent_pop = continent_pop.merge(base_pop, on='Continent')

continent_pop['Growth (%)'] = (
    (continent_pop['Population'] - continent_pop['Population_1970'])
    / continent_pop['Population_1970']
) * 100

colors = {
    'Africa': '#0072B2',
    'Asia': '#D55E00',
    'Europe': '#999999',
    'North America': '#E69F00',
    'Oceania': '#56B4E9',
    'South America': '#009E73'
}

fig = px.line(
    continent_pop,
    x='Year',
    y='Growth (%)',
    color='Continent',
    markers=True,
    color_discrete_map=colors,
    title='Africa Nearly Quadrupled Its Population Since 1970',
    labels={
        'Year': 'Year',
        'Growth (%)': 'Population growth since 1970 (%)'
    }
)

for trace in fig.data:
    trace.update(
        line=dict(width=4 if trace.name == 'Africa' else 2),
        marker=dict(size=8 if trace.name == 'Africa' else 5)
    )

latest_year = continent_pop['Year'].max()

for continent in continent_pop['Continent'].unique():
    latest = continent_pop[
        (continent_pop['Continent'] == continent) &
        (continent_pop['Year'] == latest_year)
    ]

    fig.add_scatter(
        x=latest['Year'],
        y=latest['Growth (%)'],
        mode='text',
        text=[continent],
        textposition='middle right',
        textfont=dict(color=colors[continent], size=12),
        showlegend=False,
        hoverinfo='skip'
    )

africa_growth = continent_pop[
    (continent_pop['Continent'] == 'Africa') &
    (continent_pop['Year'] == latest_year)
]['Growth (%)'].iloc[0]

fig.add_annotation(
    x=latest_year,
    y=africa_growth,
    text=f'Africa: +{africa_growth:.1f}%',
    showarrow=True,
    arrowhead=2,
    ax=-65,
    ay=-45,
    font=dict(color='#0072B2', size=13)
)

fig.update_layout(
    template='plotly_white',
    height=600,
    showlegend=False,
    hovermode='x unified',
    title_font=dict(size=21, color='#263746'),
    xaxis=dict(
        showgrid=False,
        range=[1970, 2032],
        dtick=10
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='#E5E9EF',
        zeroline=False
    ),
    margin=dict(r=130, t=100, l=70, b=60)
)

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

I chose a horizontal bar chart because it makes it easy to compare the population increase of 15 countries. I highlighted African countries in teal and used grey for the others so we can quickly see how many African countries made the list.

In [10]:
top15 = df.copy()

top15['People Added'] = (
    top15['2022 Population'] - top15['1970 Population']
)

top15 = top15.nlargest(15, 'People Added').sort_values(
    'People Added', ascending=True
)

top15['Region'] = top15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other continents'
)

top15['Added (Millions)'] = (
    top15['People Added'] / 1_000_000
)

top15['Label'] = top15['Added (Millions)'].apply(
    lambda x: f'{x:.1f}M'
)

africa_count = (top15['Continent'] == 'Africa').sum()

fig = px.bar(
    top15,
    x='People Added',
    y='Country',
    orientation='h',
    color='Region',
    color_discrete_map={
        'Africa': '#009E73',
        'Other continents': '#B8C0C8'
    },
    text='Label',
    title='India and China lead population growth, with 4 African countries in the top 15',
    labels={
        'People Added': 'Additional people (millions)',
        'Country': ''
    }
)

fig.update_traces(
    textposition='outside',
    cliponaxis=False
)

fig.update_layout(
    template='plotly_white',
    height=650,
    title_font=dict(size=19, color='#263746'),
    legend_title_text='',
    xaxis=dict(
        showgrid=True,
        gridcolor='#E5E9EF',
        showticklabels=False
    ),
    yaxis=dict(showgrid=False),
    margin=dict(l=120, r=90, t=110, b=60),
    annotations=[
        dict(
            x=0.99,
            y=1.10,
            xref='paper',
            yref='paper',
            text=f'{africa_count} of 15 countries are in Africa',
            showarrow=False,
            font=dict(size=14, color='#009E73')
        )
    ]
)

fig.show()

print(f'Number of African countries in the top 15: {africa_count}')

Number of African countries in the top 15: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

I chose a scatter plot because it helps show the relationship between a country’s land area and population. I used different colours for each continent and highlighted selected countries to show that a larger land area does not always mean a larger population.

In [11]:
colors = {
    'Africa': '#E76F51',
    'Asia': '#277DA1',
    'Europe': '#9B5DE5',
    'North America': '#F4A261',
    'South America': '#2A9D8F',
    'Oceania': '#6C757D'
}

fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    color_discrete_map=colors,
    log_x=True,
    log_y=True,
    title='Land Area Does Not Always Mean a Larger Population',
    labels={
        'Area': 'Land area (km²)',
        '2022 Population': 'Population in 2022'
    },
    opacity=0.75
)

highlight = df[df['Country'].isin([
    'Russia', 'Canada', 'Australia',
    'China', 'India', 'Bangladesh', 'Nigeria'
])]

fig.add_scatter(
    x=highlight['Area'],
    y=highlight['2022 Population'],
    mode='text',
    text=highlight['Country'],
    textposition='top center',
    textfont=dict(size=11, color='#263746'),
    showlegend=False,
    hoverinfo='skip'
)

fig.update_traces(
    marker=dict(size=10, line=dict(width=0.7, color='white')),
    selector=dict(mode='markers')
)

fig.update_layout(
    template='plotly_white',
    height=650,
    title_font=dict(size=20, color='#263746'),
    legend_title_text='Continent',
    xaxis=dict(showgrid=True, gridcolor='#E8EDF2'),
    yaxis=dict(showgrid=True, gridcolor='#E8EDF2'),
    margin=dict(l=80, r=40, t=100, b=70)
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

I chose a scatter plot because it helps show the relationship between a country’s land area and population. I used different colours for each continent and highlighted selected countries to show that a larger land area does not always mean a larger population.

In [12]:
colors = {
    'Africa': '#E76F51',
    'Asia': '#277DA1',
    'Europe': '#9B5DE5',
    'North America': '#F4A261',
    'South America': '#2A9D8F',
    'Oceania': '#6C757D'
}

fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    color_discrete_map=colors,
    log_x=True,
    log_y=True,
    title='Land Area Does Not Always Mean a Larger Population',
    labels={
        'Area': 'Land area (km²)',
        '2022 Population': 'Population in 2022'
    },
    opacity=0.75
)

highlight = df[df['Country'].isin([
    'Russia', 'Canada', 'Australia',
    'China', 'India', 'Bangladesh', 'Nigeria'
])]

fig.add_scatter(
    x=highlight['Area'],
    y=highlight['2022 Population'],
    mode='text',
    text=highlight['Country'],
    textposition='top center',
    textfont=dict(size=11, color='#263746'),
    showlegend=False,
    hoverinfo='skip'
)

fig.update_traces(
    marker=dict(size=10, line=dict(width=0.7, color='white')),
    selector=dict(mode='markers')
)

fig.update_layout(
    template='plotly_white',
    height=650,
    title_font=dict(size=20, color='#263746'),
    legend_title_text='Continent',
    xaxis=dict(showgrid=True, gridcolor='#E8EDF2'),
    yaxis=dict(showgrid=True, gridcolor='#E8EDF2'),
    margin=dict(l=80, r=40, t=100, b=70)
)

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

I chose a choropleth map because it shows population growth and decline by country on a world map. I used blue for growth and orange for decline so the differences are easy to identify.

In [14]:
map_df = df.copy()

map_df['Population Change (%)'] = (
    (map_df['2022 Population'] - map_df['2000 Population'])
    / map_df['2000 Population']
) * 100

growth_count = (map_df['Population Change (%)'] > 0).sum()
decline_count = (map_df['Population Change (%)'] < 0).sum()

fig = px.choropleth(
    map_df,
    locations='CCA3',
    color='Population Change (%)',
    hover_name='Country',
    hover_data={
        'Population Change (%)': ':.1f',
        'CCA3': False
    },
    color_continuous_scale=[
        [0, '#D97732'],
        [0.45, '#F1D8B8'],
        [0.5, '#F4F4F0'],
        [0.55, '#C6DCE8'],
        [1, '#176B87']
    ],
    color_continuous_midpoint=0,
    title='Population Growth and Decline Across the World (2000–2022)',
    labels={
        'Population Change (%)': 'Population change (%)'
    }
)

fig.update_geos(
    projection_type='natural earth',
    showcoastlines=True,
    coastlinecolor='white',
    showframe=False,
    showocean=True,
    oceancolor='#EAF0F4',
    showland=True,
    landcolor='#F4F4F0',
    showcountries=True,
    countrycolor='white'
)

fig.update_layout(
    template='plotly_white',
    height=600,
    title_font=dict(size=20, color='#263746'),
    coloraxis_colorbar=dict(
        title='Change (%)',
        ticksuffix='%'
    ),
    margin=dict(l=10, r=10, t=100, b=30),
    annotations=[
        dict(
            x=0.5,
            y=1.08,
            xref='paper',
            yref='paper',
            text=f'{growth_count} countries grew | {decline_count} countries declined',
            showarrow=False,
            font=dict(size=14, color='#176B87')
        )
    ]
)

fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

I chose a treemap because it shows continents and their countries together in one chart. The size of each rectangle represents its population, while different colours help separate the continents.

In [15]:
treemap_df = df.copy()

continent_totals = treemap_df.groupby(
    'Continent'
)['2022 Population'].sum()

world_total = treemap_df['2022 Population'].sum()

asia_share = (
    continent_totals['Asia'] / world_total
) * 100

colors = {
    'Africa': '#E76F51',
    'Asia': '#277DA1',
    'Europe': '#9B5DE5',
    'North America': '#F4A261',
    'South America': '#2A9D8F',
    'Oceania': '#8D99AE'
}

fig = px.treemap(
    treemap_df,
    path=[px.Constant('World'), 'Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    color_discrete_map=colors,
    hover_data={
        '2022 Population': ':,',
        'Continent': True
    },
    title='Asia Holds 59.2% of the World’s Population in 2022'
)

fig.update_traces(
    textinfo='label+percent parent',
    marker=dict(
        line=dict(color='white', width=2)
    ),
    root_color='#263746',
    tiling=dict(pad=3),
    textfont=dict(size=12)
)

fig.update_layout(
    template='plotly_white',
    height=700,
    title_font=dict(size=20, color='#263746'),
    margin=dict(t=100, l=15, r=15, b=20),
    uniformtext=dict(minsize=9, mode='hide')
)

fig.show()

print(f'World population: {world_total:,}')
print(f'Asia share of world population: {asia_share:.1f}%')

World population: 7,973,413,042
Asia share of world population: 59.2%


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

I chose a 100% stacked bar chart because it shows how each continent’s share of the world population changed over time. I grouped the three smaller continents together and used different colours to make the changes easier to compare.

In [17]:
q7_df = df_long.copy()

q7_df['Group'] = q7_df['Continent'].replace({
    'North America': 'Rest of the world',
    'South America': 'Rest of the world',
    'Oceania': 'Rest of the world'
})

q7_grouped = q7_df.groupby(
    ['Year', 'Group'], as_index=False
)['Population'].sum()

q7_grouped['World Total'] = q7_grouped.groupby(
    'Year'
)['Population'].transform('sum')

q7_grouped['World Share (%)'] = (
    q7_grouped['Population'] / q7_grouped['World Total']
) * 100

group_order = [
    'Africa',
    'Asia',
    'Europe',
    'Rest of the world'
]

colors = {
    'Africa': '#009E73',
    'Asia': '#277DA1',
    'Europe': '#9B5DE5',
    'Rest of the world': '#C7CDD3'
}

fig = px.bar(
    q7_grouped,
    x='Year',
    y='World Share (%)',
    color='Group',
    category_orders={'Group': group_order},
    color_discrete_map=colors,
    barmode='relative',
    title='Africa Gains World Population Share While Europe Loses Ground',
    labels={
        'Year': 'Year',
        'World Share (%)': 'Share of world population (%)',
        'Group': ''
    }
)

fig.update_layout(
    template='plotly_white',
    height=600,
    title_font=dict(size=20, color='#263746'),
    barnorm='percent',
    yaxis=dict(
        range=[0, 100],
        ticksuffix='%',
        showgrid=True,
        gridcolor='#E5E9EF'
    ),
    xaxis=dict(
        tickmode='array',
        tickvals=[1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022],
        showgrid=False
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5
    ),
    margin=dict(l=70, r=40, t=120, b=60)
)

fig.show()
fig.write_image("/Users/gvsujith/Desktop/dataviz-exercises-sujith/quiz/charts/Q7_population_share.png", scale=2)